In [2]:
import sys
sys.path.append("..")
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from src.features import make_features

df_clean = pd.read_parquet("../data/processed/electricity_clean.parquet")
temp = pd.read_parquet("../data/processed/air_temperature.parquet")
train_buildings = pd.read_csv("../configs/train_buildings.csv")["building"].tolist()

rng = np.random.default_rng(0)
sample = rng.choice(train_buildings, size=100, replace=False)
data = pd.concat([make_features(b, df_clean, temp) for b in sample])
data = data[data.index.year == 2016]        # step 4.1 uses 2016 only; 2017 stays untouched
site = data["building"].str.split("_").str[0].to_numpy()

h, wd = data["hour"].to_numpy(), data["weekday"].to_numpy()
X = np.column_stack([
    np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24),
    np.sin(2 * np.pi * wd / 7), np.cos(2 * np.pi * wd / 7),
    data["lag24_n"].clip(upper=5).to_numpy(),
    data["lag168_n"].clip(upper=5).to_numpy(),
    data["temp"].to_numpy(),
]).astype("float32")
y = data["load_n"].clip(upper=5).to_numpy(dtype="float32")

ok = ~np.isnan(X).any(axis=1) & ~np.isnan(y)
month = data.index.month
is_fit = ok & (month <= 10)
is_val = ok & (month >= 11)
sites = sorted(set(site[is_fit]))

info = pd.DataFrame({"buildings": data[is_fit].groupby(site[is_fit])["building"].nunique(),
                     "fit_rows": pd.Series(site[is_fit]).value_counts(),
                     "val_rows": pd.Series(site[is_val]).value_counts()}).sort_values("fit_rows")
print(len(sites), "sites (clients),", is_fit.sum(), "fit rows,", is_val.sum(), "validation rows")
print(info)

12 sites (clients), 673104 fit rows, 138859 validation rows
          buildings  fit_rows  val_rows
Peacock           2     14266      2855
Moose             3     19232      3320
Gator             5     31197      4536
Bear              5     34760      6936
Robin             6     38424      8766
Wolf              6     42735      8760
Cockatoo          8     44360      9256
Eagle             8     51534     11120
Fox               9     63485     13167
Bull             12     78514     17544
Hog              14     98438     20479
Rat              22    156159     32120


In [3]:
stats = []
for s in sites:
    Xs = X[is_fit & (site == s), 4:].astype("float64")
    stats.append((len(Xs), Xs.sum(axis=0), (Xs ** 2).sum(axis=0)))   # what each client sends

n = sum(c for c, _, _ in stats)
mu = sum(t for _, t, _ in stats) / n
sd = np.sqrt(sum(q for _, _, q in stats) / n - mu ** 2)

pooled = X[is_fit, 4:].astype("float64")
print("max difference vs pooled mean:", np.abs(mu - pooled.mean(axis=0)).max())
print("max difference vs pooled std: ", np.abs(sd - pooled.std(axis=0)).max())
X[:, 4:] = ((X[:, 4:] - mu) / sd).astype("float32")

max difference vs pooled mean: 1.1102230246251565e-16
max difference vs pooled std:  5.309530592967349e-12


In [4]:
def make_mlp():
    return nn.Sequential(nn.Linear(7, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 1))


def predict(model, Xt):
    model.eval()
    with torch.no_grad():
        out = [model(Xt[i:i + 100_000]) for i in range(0, len(Xt), 100_000)]
    return torch.cat(out).squeeze(1).numpy()


def run_epoch(model, opt, Xtr, ytr, gen):
    """One pass over (Xtr, ytr) in shuffled batches of 1024."""
    model.train()
    loss_fn = nn.MSELoss()
    perm = torch.randperm(len(Xtr), generator=gen)
    for i in range(0, len(Xtr), 1024):
        idx = perm[i:i + 1024]
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()


def tensors(mask):
    return torch.from_numpy(X[mask]), torch.from_numpy(y[mask]).unsqueeze(1)


clients = {s: tensors(is_fit & (site == s)) for s in sites}
Xf, yf = tensors(is_fit)
Xv, yv = torch.from_numpy(X[is_val]), y[is_val]
site_v = site[is_val]


def val_mse(model):
    """Validation MSE overall and per site."""
    se = pd.Series((predict(model, Xv) - yv) ** 2)
    return se.mean(), se.groupby(site_v).mean()

In [ ]:
torch.manual_seed(0)
central = make_mlp()
init_state = copy.deepcopy(central.state_dict())   # FedAvg starts from the same weights
gen = torch.Generator().manual_seed(0)
opt = torch.optim.Adam(central.parameters(), lr=1e-3)

curve_c, site_c = [], []
for epoch in range(1, 16):
    run_epoch(central, opt, Xf, yf, gen)
    overall, by_site = val_mse(central)
    curve_c.append(overall)
    site_c.append(by_site)
    print("epoch", epoch, "validation MSE", round(overall, 4))

epoch 1 validation MSE 0.037
epoch 2 validation MSE 0.0367
epoch 3 validation MSE 0.037
